# PINN vs VPINN — transient heat conduction with a nonlocal integral boundary condition

Short version of the benchmark in
[`transient-heat-nonlocal-bc`](https://github.com/MaximeAuger/transient-heat-nonlocal-bc).

$$u_t - \bigl(k(u)\,u_x\bigr)_x = f(x,t), \qquad k(u) = 1 + \beta u^2, \qquad x \in (0,1),\ t \in (0,T]$$

$$u(0,t) = \alpha \int_0^1 u(x,t)\,dx + g_0(t), \qquad u(1,t) = g_1(t), \qquad u(x,0) = u_0(x)$$

The notebook (a few minutes on a Colab CPU):

1. checks the manufactured solution and the critical value $\alpha^* = 2$,
2. trains a PINN and a VPINN on the linear case with a **reduced budget**,
3. compares the space-time errors and the boundary flux $\lambda(t)$ learned by the VPINN,
4. runs one nonlinear case.

The full study (3 seeds, all sweeps and ablations) is `python src/run_all.py`.

In [ ]:
import os
if not os.path.isdir("transient-heat-nonlocal-bc"):
    !git clone -q https://github.com/MaximeAuger/transient-heat-nonlocal-bc.git
%cd transient-heat-nonlocal-bc/src

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from exact_solution import manufactured, verify
from critical_alpha import leading_eigenvalue, critical_alpha_numerical, symbolic_derivation
from fd_reference import fd_errors
from pinn_solver import train_pinn
from vpinn_solver import train_vpinn

C_PINN, C_VPINN = "#2a78d6", "#eb6834"

## 1. Verification before any training

The data $f, g_0, g_1, u_0$ are derived with SymPy from the manufactured solution
$u = e^{-t}\,[\sin(\pi x) + (1-x)\,4/\pi]$. The residual is checked independently with
PyTorch automatic differentiation.

In [ ]:
for alpha, beta in [(1.0, 0.0), (1.0, 2.0), (2.5, 0.0)]:
    verify(manufactured("decay", alpha, beta))
    print()

**Critical value of $\alpha$.** For $u_t = u_{xx}$, $u(1,t)=0$, $u(0,t)=\alpha\int_0^1 u\,dx$,
the mode $e^{\kappa^2 t}\sinh(\kappa(1-x))$ exists iff $\alpha = \kappa\coth(\kappa/2) > 2$:
the problem has an exponentially growing mode exactly when $\alpha > \alpha^* = 2$.

In [ ]:
sym = symbolic_derivation()
print("zero mode phi = 1 - x requires alpha =", sym["alpha_zero_mode"])
print("growing mode: alpha =", sym["alpha_growing"], " -> limit kappa->0:", sym["alpha_limit"])
print("alpha* from the spectrum of the FD operator:", critical_alpha_numerical(200))
for a in (1.0, 1.99, 2.5, 3.0):
    print(f"alpha = {a:4.2f}:  leading eigenvalue sigma_1 = {leading_eigenvalue(a):+.4f}")

## 2. Linear case ($\beta = 0$, $\alpha = 1$), reduced budget

Both methods share the network, the Gauss–Legendre space-time grid, the hard-imposed
initial and Dirichlet conditions, the penalty on the nonlocal condition and the optimiser
(Adam then L-BFGS). They differ only in how the PDE residual is measured.

In [ ]:
budget = {"n_adam": 1500, "n_lbfgs": 1500}   # the full benchmark uses 4000 + 4000
pinn = train_pinn({**budget}, verbose=False)
vpinn = train_vpinn({**budget}, verbose=False)
fd = fd_errors(manufactured("decay"), 200, 200)

print(f"{'':28s}{'PINN':>12s}{'VPINN':>12s}{'FD 200x200':>12s}")
for name, key in [("L2 error (space-time)", "L2"), ("Linf error", "Linf"),
                  ("H1 error at t = T", "H1_T"), ("nonlocal condition (max)", "nonlocal_max")]:
    f = f"{fd[key]:12.2e}" if key in fd else f"{'-':>12s}"
    print(f"{name:28s}{pinn['errors'][key]:12.2e}{vpinn['errors'][key]:12.2e}{f}")
print(f"{'flux, autograd (rel. L2)':28s}{pinn['errors']['flux_ad_rel']:12.2e}{vpinn['errors']['flux_ad_rel']:12.2e}")
print(f"{'flux, learned lambda(t)':28s}{'-':>12s}{vpinn['errors']['flux_lam_rel']:12.2e}")
print(f"{'training time (s)':28s}{pinn['time']:12.1f}{vpinn['time']:12.1f}")

In [ ]:
from matplotlib.colors import LogNorm
ep, ev = pinn["errors"], vpinn["errors"]
vmax = max(np.abs(ep["error"]).max(), np.abs(ev["error"]).max())
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
im = axes[0].pcolormesh(ep["t"], ep["x"], ep["u_ref"], cmap="Blues", shading="auto")
fig.colorbar(im, ax=axes[0], label="u"); axes[0].set_title("Exact solution")
for ax, e, name in ((axes[1], ep, "PINN"), (axes[2], ev, "VPINN")):
    im = ax.pcolormesh(e["t"], e["x"], np.maximum(np.abs(e["error"]), vmax * 1e-3),
                       cmap="Blues", norm=LogNorm(vmax * 1e-3, vmax), shading="auto")
    fig.colorbar(im, ax=ax, label="|error|"); ax.set_title(f"{name} error (max {np.abs(e['error']).max():.1e})")
for ax in axes:
    ax.set_xlabel("t"); ax.set_ylabel("x")
plt.tight_layout(); plt.show()

## 3. Boundary flux: the Lagrange multiplier $\lambda(t)$

In the weak form the flux through the nonlocal boundary, $\lambda(t) = -k(u)\,u_x(0,t)$,
appears as an unknown function of time, expanded on Legendre polynomials with trainable
coefficients.

In [ ]:
t = ev["t"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(t, ev["flux_ref"], color="0.2", lw=4, alpha=0.3, label="exact")
axes[0].plot(t, ev["flux_lam"], color=C_VPINN, label="VPINN learned λ(t)")
axes[0].plot(t, ep["flux_ad"], color=C_PINN, ls="--", label="PINN autograd flux")
axes[0].set_xlabel("t"); axes[0].set_ylabel("flux at x = 0"); axes[0].legend()
axes[1].semilogy(t, np.abs(ev["flux_lam"] - ev["flux_ref"]), color=C_VPINN, label="VPINN learned λ(t)")
axes[1].semilogy(t, np.abs(ep["flux_ad"] - ep["flux_ref"]), color=C_PINN, ls="--", label="PINN autograd flux")
axes[1].set_xlabel("t"); axes[1].set_ylabel("|flux error|"); axes[1].legend()
plt.tight_layout(); plt.show()
print("Legendre coefficients of λ(t):", np.round(vpinn["lam_coeffs"], 5))

## 4. One nonlinear case ($\beta = 2$)

In [ ]:
pinn_nl = train_pinn({**budget, "beta": 2.0}, verbose=False)
vpinn_nl = train_vpinn({**budget, "beta": 2.0}, verbose=False)
print(f"beta = 2:  PINN L2 = {pinn_nl['errors']['L2']:.2e}   VPINN L2 = {vpinn_nl['errors']['L2']:.2e}")
print(f"           VPINN learned flux, relative L2 error = {vpinn_nl['errors']['flux_lam_rel']:.2e}")

## 5. Going further

```bash
python run_all.py --quick     # every experiment with a tiny budget (a few minutes)
python run_all.py             # full benchmark: 3 seeds, E1-E4, tables and figures in results/
```

A single run of this notebook uses one seed and a reduced budget: the numbers above are
indicative only. The tables of the repository report the mean ± std over 3 seeds.